In [ ]:
import pandas as pd
import numpy as np
import re
import os

In [ ]:
# Set benchmark name. Options are recon and flask.
bench_name="recon" #flask

In [ ]:
eval_folder_path = os.path.dirname(os.path.dirname(os.getcwd()))

#Prediction path
path=eval_folder_path+f"/evaluation_results/{bench_name}/"
l=os.listdir(path)
l.sort()

#Labels
gold=pd.read_json(eval_folder_path+f"/benchmarks/{bench_name}/{bench_name}_en.jsonl",lines=True)

print("Files to evaluate: ")
display(l)

## 1. Overall model performance

In [ ]:
def extract_value(text, lax=True, print_no_match=False):
    """Extracts the score. If lax=True, it will also consider outputs that do not match the prompted format but output a score."""

    pattern = r"\[(RESULT|EMAITZA|RESULTADO)\]\s*([1-5])"
    match = re.search(pattern, text)

    test=re.search(r"\[FORMAT ERROR\]\s*([1-5])", text)
    if match:
        return int(match.group(2).strip())
    elif test and lax:
        return int(test.group(1).strip())
    if print_no_match:
        print(text,"\n\n")
    return 0

In [ ]:
from sklearn.metrics import mean_squared_error
from scipy import stats
import krippendorff

def exact_match(pred,label):
    return (np.mean([1 if x==y else 0 for x,y in zip(pred,label)]))

def get_correlations(l,gold,print_results=False,mean_dict=False):

    result_dict={}
    for i,doc in enumerate(l):
        print(f"Processing document {i+1}/{len(l)}",end="\r")
        train_lang=doc.split("-")[0]
        aux=doc.split(f"_{bench_name}_")
        test_lang=aux[-1][:-6]
        model_name=aux[0][len(train_lang)+1:]

        if print_results:
            print(doc)
        file_result_dict={}

        d=pd.read_json(path+doc,lines=True).loc[gold.index]
        
        try:
            d["scores"]=d.apply(lambda x: [extract_value(x["outputs_0"]),extract_value(x["outputs_1"]),extract_value(x["outputs_2"])], axis=1,)
        except:
            print("Single prediction in", doc)
            d["scores"]=d.apply(lambda x: extract_value(x["outputs_0"]), axis=1)

        label = gold["gold"].to_list()
        pred=d.apply(lambda x:stats.mode(x["scores"], keepdims=False).mode, axis=1,).to_list()

        #-------------Calculate metrics-------------
        #Pearson correlation
        pearson=stats.pearsonr(pred,label)
        pears=pearson.statistic
        p_val=pearson.pvalue

        #Exact match
        em=exact_match(pred,label)
        #Mean Squared Error
        mse=mean_squared_error(label,pred)
        #Krippendorff's alpha. Only considered if there are multiple predictions per item, otherwise nan.
        try:
            k_alpha=krippendorff.alpha(reliability_data=np.array(d["scores"].to_list()).T, level_of_measurement='nominal')
        except:
            k_alpha=np.nan
        #Missing predictions
        missing=pred.count(0)

        #---------------Save results----------------
        file_result_dict["model"]=model_name
        file_result_dict["train_lang"]=train_lang
        file_result_dict["pearson"]=pears
        file_result_dict["p_val"]=p_val
        file_result_dict["exact_match"]=em
        file_result_dict["mse"]=mse
        if not np.isnan(k_alpha):
            file_result_dict["krippendorff_alpha"]=k_alpha
        file_result_dict["missing_predictions"]=missing

        try:
            result_dict[test_lang].append(file_result_dict)
        except:
            result_dict[test_lang]=[file_result_dict]

        #---------------Print results----------------
        if print_results:
            print("Correlations")
            print("Pearson",round(pears,3), "   (p-value of", round(p_val,3),")")
            print("Exact Match",round(em,3))
            print("MSE",round(mse,3))
            if not np.isnan(k_alpha):
                print("krippendorffs alpha",round(k_alpha,3))
            print(f"Missing predictions (count of 0 s): {missing}")
            print("\n")
    if print_results:
        print("___________________________________________________________________________________________________")

    if mean_dict:
        """Only used in ./flask_criteria.ipynb to average the results per criteria. Defaulted to False."""
        import numbers

        keys = list(result_dict.keys())
        n = len(result_dict[keys[0]])

        mean_list = []

        for i in range(n):
            dicts = [result_dict[k][i] for k in keys]
            out = {}
            
            for field in dicts[0]:
                vals = [d[field] for d in dicts]
                
                if isinstance(vals[0], numbers.Number):
                    out[field] = sum(vals) / len(vals)
                else:
                    out[field] = vals[0]  # keep string
            
            mean_list.append(out)

        return(mean_list)
    else:
        return result_dict

In [ ]:
results=get_correlations(l,gold)
results